In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "gold")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
# --- STEP 1: WIDGETS (Parameterization) ---
dbutils.widgets.text("catalog_param", "my_assessment", "1. Catalog")
dbutils.widgets.text("schema_param", "gold", "2. Schema")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")
target_table = "revenue_by_nation"
full_table_path = f"{catalog}.{schema}.{target_table}"

# --- STEP 2: LIBRARIES & DATA LOADING ---
from pyspark.sql.functions import sum, broadcast, col

# Reading from Silver (Assuming these exist from your previous parallel step)
customer_df = spark.table(f"{catalog}.silver.customer_cleaned")
nation_df = spark.table(f"{catalog}.silver.nation_cleaned")
orders_df = spark.table(f"{catalog}.silver.orders_cleaned")

# --- STEP 3: SPARK OPTIMIZATIONS (Broadcast Join) ---
# Small table (nation) is broadcasted to avoid shuffle
customer_with_nation = customer_df.join(
    broadcast(nation_df), 
    customer_df.c_nationkey == nation_df.n_nationkey
)

# Join with orders
final_joined_df = customer_with_nation.join(
    orders_df, 
    customer_with_nation.c_custkey == orders_df.o_custkey
)

# --- STEP 4: GOLD LOGIC (Aggregation) ---
gold_revenue_df = final_joined_df.groupBy("n_name") \
    .agg(sum("o_totalprice").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

# --- STEP 5: WRITE WITH LIQUID CLUSTERING ---
# This saves the data and defines the clustering key in one go
gold_revenue_df.write.mode("overwrite") \
    .option("clusterBy", "n_name") \
    .saveAsTable(full_table_path)

print(f"✅ Table {full_table_path} created successfully.")

# --- STEP 6: MAINTENANCE (Optimize & Vacuum) ---
# We use spark.sql to run these maintenance commands using the widget variables
spark.sql(f"OPTIMIZE {full_table_path}")
spark.sql(f"VACUUM {full_table_path} RETAIN 168 HOURS")

print(f"🚀 Optimization and Vacuum complete for {full_table_path}")